In [0]:
%run ../../commonsetup/Setup-Common

### Silver: streaming Trip Updates
##### Unlike vehicle_positions, Bronze already stores this typed/structured (no raw_json/from_json step). The real work here is exploding stop_time_updates down to one row per (trip, stop) -- the grain delay metrics actually need, not one row per trip.
##### Append-only, no MERGE: the checkpoint + Delta's transactional write already make re-runs idempotent, and each poll is an immutable observation with nothing to overwrite.

In [0]:
from pyspark.sql import functions as F


### Config

In [0]:
ENTITY                  = "trip_updates"

BRONZE_TABLE           = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_gtfs_{ENTITY}"
SILVER_TABLE           = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_{ENTITY}"
QUARANTINE_TABLE       = SILVER_TABLE + "_quarantine"
CHECKPOINT_HISTORY     = f"{CHECKPOINT_BASE}/silver_gtfs_{ENTITY}/checkpoint"
CHECKPOINT_QUARANTINE  = f"{CHECKPOINT_BASE}/silver_gtfs_{ENTITY}_quarantine/checkpoint"



### Explode to (trip, stop) grain
##### explode_outer, not explode -- a trip with an empty stop_time_updates array should still get a row (nulls for the stop fields), not vanish from Silver.

In [0]:
bronze_df = spark.readStream.table(BRONZE_TABLE)

parsed_df = (
    bronze_df
    .withColumn("stu", F.explode_outer("stop_time_updates"))
    .select(
        F.col("gtfs_mode"),
        F.col("entity_id"),
        F.col("trip_id"),
        F.col("route_id"),
        F.col("start_date"),
        F.col("vehicle_id"),
        F.col("poll_timestamp").cast("timestamp").alias("poll_timestamp"),
        F.col("ingestion_timestamp").cast("timestamp").alias("ingestion_timestamp"),
        F.col("stu.stop_sequence").alias("stop_sequence"),
        F.col("stu.stop_id").alias("stop_id"),
        F.col("stu.arrival_delay").alias("arrival_delay_seconds"),
        F.col("stu.departure_delay").alias("departure_delay_seconds"),
    )
)



In [0]:
# Can't attribute to a trip/stop -> quarantine (same principle as vehicle_positions).
good_df = parsed_df.filter(
    F.col("trip_id").isNotNull()
    & (F.col("stop_sequence").isNotNull() | F.col("stop_id").isNotNull())
)
bad_df = parsed_df.filter(
    F.col("trip_id").isNull()
    | (F.col("stop_sequence").isNull() & F.col("stop_id").isNull())
)



In [0]:
# ---- Write: append-only Silver history ------------------------------------
history_query = (
    good_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_HISTORY)
    .partitionBy("gtfs_mode")
    .trigger(availableNow=True)          # swap to .trigger(processingTime="30 seconds") for a continuously running job
    .toTable(SILVER_TABLE)
)

# ---- Write: quarantine ------------------------------------------------------
quarantine_query = (
    bad_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_QUARANTINE)
    .trigger(availableNow=True)
    .toTable(QUARANTINE_TABLE)
)

history_query.awaitTermination()
quarantine_query.awaitTermination()

### Next: Gold
##### Stays a separate Silver fact from vehicle_positions -- different grain. Gold joins the two plus the static dims.

In [0]:
%sql
select * from silver_gtfs_trip_updates limit 20;